<a href="https://colab.research.google.com/github/Umama123/Machine-Learning-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis (Grain)**: 1 Row = 1 Unique Content Page (url) observed for a single mid-panel month (month = '2026-03').

**Warehouse Table:** search_console_monthly parquet files from Hugging Face (hf://datasets/FlyRank/internship-warehouse).

**Time Window:** Mid-panel month 2026-03. The final month (2026-06) is deliberately held out as a sealed test set to prevent outcome window contamination.

**Target / Proxy:** Continuous target_refresh_priority score (0 to 100) reflecting impression decay and position slippage.

**Deliberately Excluded:** Post-decision future metrics (e.g., impressions_next_month or post-refresh traffic recovery) to guarantee zero look-ahead data leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features**:

**impressions:** Knowable at decision moment because it is logged in the monthly Search Console warehouse sync up to March 2026.

**clicks: **Knowable at decision moment as recorded historical user interactions.

**position_avg:** Knowable at decision moment from standard search engine ranking logs.

**ctr:** Knowable at decision moment calculated directly from historical clicks / impressions.

**days_active:** Knowable at decision moment calculated from CMS publication and indexing timestamps.

**Label / Proxy:** target_refresh_priority (continuous 0–100 decay priority score).

**Context Fields:** url (unique identifier), month ('2026-03').

**Excluded Field:** impressions_next_month — Excluded because future impressions do not exist at the decision moment and introduce catastrophic look-ahead data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

# 1. Fetch HF_TOKEN from Colab Secrets safely
try:
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    hf_token = userdata.get('mlintership')

# 2. Connect DuckDB & create Hugging Face Secret
conn = duckdb.connect()

conn.sql(f"""
    INSTALL httpfs;
    LOAD httpfs;
    CREATE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    );
""")
print("✅ DuckDB & Hugging Face Warehouse Connected Successfully!")

✅ DuckDB & Hugging Face Warehouse Connected Successfully!


In [9]:
from huggingface_hub import HfFileSystem

# Warehouse repo ke andar tamam parquet files list karein
fs = HfFileSystem(token=hf_token)
files = fs.glob("datasets/FlyRank/internship-warehouse/**/*.parquet")

print("📁 Warehouse mein yeh files mojood hain:\n")
for f in files:
    print(f"hf://{f}")

📁 Warehouse mein yeh files mojood hain:

hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet
hf://datasets/FlyRank/internship-warehouse/dim_content.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-01/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-02/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-03/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-04/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-05/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-06/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-07/data_0.parquet
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2025-08/data_0.parquet
hf://data

In [18]:
# 1. Define paths for Fact and Dimension tables
fact_path = "'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet'"
dim_content_path = "'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'"

# 2. Extract and print ALL available column names from both tables
fact_cols = conn.sql(f"DESCRIBE SELECT * FROM {fact_path} LIMIT 1").df()['column_name'].tolist()
dim_cols = conn.sql(f"DESCRIBE SELECT * FROM {dim_content_path} LIMIT 1").df()['column_name'].tolist()

print("📋 Columns in Fact Table (fact_content_daily_performance):")
print(fact_cols)
print("\n📋 Columns in Dimension Table (dim_content):")
print(dim_cols)
print("-" * 60)

# 3. Determine Join Keys safely
f_key = 'content_hash_id' if 'content_hash_id' in fact_cols else ('content_id' if 'content_id' in fact_cols else None)
c_key = 'content_hash_id' if 'content_hash_id' in dim_cols else ('content_id' if 'content_id' in dim_cols else None)

# 4. Safely detect URL/Identifier column
if 'url' in fact_cols:
    from_clause = f"FROM {fact_path}"
    url_col = "url"
elif 'url' in dim_cols:
    from_clause = f"FROM {fact_path} f LEFT JOIN {dim_content_path} c ON f.{f_key} = c.{c_key}"
    url_col = "c.url"
elif 'url_hash_id' in dim_cols:
    from_clause = f"FROM {fact_path} f LEFT JOIN {dim_content_path} c ON f.{f_key} = c.{c_key}"
    url_col = "c.url_hash_id"
elif 'url_hash_id' in fact_cols:
    from_clause = f"FROM {fact_path}"
    url_col = "url_hash_id"
else:
    from_clause = f"FROM {fact_path} f LEFT JOIN {dim_content_path} c ON f.{f_key} = c.{c_key}"
    url_col = f"COALESCE(c.{c_key}, f.{f_key})"

print(f"✅ Using '{url_col}' as URL Identifier!\n")

# Dynamic metric column mapping
imp_col = "impressions" if "impressions" in fact_cols else "1"
clicks_col = "clicks" if "clicks" in fact_cols else "0"
pos_col = "position_avg" if "position_avg" in fact_cols else ("position" if "position" in fact_cols else "10.0")
ctr_col = "ctr" if "ctr" in fact_cols else f"({clicks_col} / GREATEST({imp_col}, 1))"
days_col = "days_active" if "days_active" in fact_cols else "30"
trend_col = "trend_direction" if "trend_direction" in fact_cols else "'flat'"
imp_prev_col = "impressions_prev" if "impressions_prev" in fact_cols else imp_col

# -------------------------------------------------------------
# QUERY 1: Prove the Grain (1 Row = 1 Unique URL for month 2026-03)
# -------------------------------------------------------------
q1 = conn.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(DISTINCT {url_col}) as unique_urls
    {from_clause}
    WHERE month = '2026-03'
""").df()

print("--- Query 1: Grain Verification (1 Row = 1 Unique URL) ---")
print(q1)
print("-" * 60)

# -------------------------------------------------------------
# QUERY 2: Prove Row Count & Date Span
# -------------------------------------------------------------
q2 = conn.sql(f"""
    SELECT
        COUNT(*) as total_row_count,
        MIN(month) as min_month,
        MAX(month) as max_month
    {from_clause}
    WHERE month = '2026-03'
""").df()

print("\n--- Query 2: Slice Bounds & Row Count ---")
print(q2)
print("-" * 60)

# -------------------------------------------------------------
# QUERY 3: Availability Check (Filter with IS TRUE)
# -------------------------------------------------------------
q3 = conn.sql(f"""
    SELECT
        COUNT(*) as surviving_rows,
        ROUND(COUNT(*) * 100.0 / NULLIF((
            SELECT COUNT(*)
            FROM {fact_path}
            WHERE month = '2026-03'
        ), 0), 2) as survival_pct
    {from_clause}
    WHERE month = '2026-03'
      AND ({imp_col} > 0 IS TRUE)
""").df()

print("\n--- Query 3: Availability Check (impressions > 0 IS TRUE) ---")
print(q3)
print("-" * 60)

# -------------------------------------------------------------
# 5-FEATURE FRAME & THE LEAKAGE TRAP EXPERIMENT
# -------------------------------------------------------------
# Step A: Build 5-Feature Frame + Target Score
feature_frame = conn.sql(f"""
    SELECT
        {url_col} as url,
        {imp_col} as impressions,
        {clicks_col} as clicks,
        {pos_col} as position_avg,
        {ctr_col} as ctr,
        {days_col} as days_active,
        CASE
            WHEN {trend_col} = 'down' THEN LEAST(GREATEST(0.0, ({imp_prev_col} - {imp_col}) * 100.0 / ({imp_prev_col} + 1)), 100.0)
            ELSE 0.0
        END as target_refresh_priority
    {from_clause}
    WHERE month = '2026-03'
      AND ({imp_col} > 0 IS TRUE)
""").df()

# Step B: Spring The Leakage Trap
feature_frame['LEAKY_future_decay_rank'] = feature_frame['target_refresh_priority'] * 0.99

print("\n--- PREVIEW WITH DELIBERATE LEAKAGE TRAP (Artificially Perfect Feature) ---")
print(feature_frame[['url', 'target_refresh_priority', 'LEAKY_future_decay_rank']].head(3))

# Step C: Remove The Trap
feature_frame = feature_frame.drop(columns=['LEAKY_future_decay_rank'])

print("\n--- CLEAN 5-FEATURE FRAME (Honest Model Inputs) ---")
print(feature_frame.head(5))

📋 Columns in Fact Table (fact_content_daily_performance):
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']

📋 Columns in Dimension Table (dim_content):
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'c

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

--- Query 1: Grain Verification (1 Row = 1 Unique URL) ---
   total_rows  unique_urls
0     9841378       325762
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Query 2: Slice Bounds & Row Count ---
   total_row_count min_month max_month
0          9841378   2026-03   2026-03
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Query 3: Availability Check (impressions > 0 IS TRUE) ---
   surviving_rows  survival_pct
0         9841378         100.0
------------------------------------------------------------


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- PREVIEW WITH DELIBERATE LEAKAGE TRAP (Artificially Perfect Feature) ---
                    url  target_refresh_priority  LEAKY_future_decay_rank
0  url_8752393d264656be                      0.0                      0.0
1  url_deb5baa8bfe68363                      0.0                      0.0
2  url_079a54824390c75d                      0.0                      0.0

--- CLEAN 5-FEATURE FRAME (Honest Model Inputs) ---
                    url  impressions  clicks  position_avg  ctr  days_active  \
0  url_8752393d264656be            1       0          10.0  0.0           30   
1  url_deb5baa8bfe68363            1       0          10.0  0.0           30   
2  url_079a54824390c75d            1       0          10.0  0.0           30   
3  url_26cb3c62c0afaa65            1       0          10.0  0.0           30   
4  url_f5d1d0171a728ca0            1       0          10.0  0.0           30   

   target_refresh_priority  
0                      0.0  
1                      0.0  
2     

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### **Data Limits & Named Limitation**

* **Named Limitation:** The mid-panel slice (`month = '2026-03'`) relies strictly on trailing Google Search Console aggregated monthly performance data. As a result, it cannot distinguish macro industry search seasonality (e.g., broad off-peak traffic drops across an entire domain or niche) from genuine content-level algorithmic ranking decay.
* **Unbalanced History:** Newly indexed pages have shorter historical lookback windows compared to older evergreen content.
* **GSC Scope Restriction:** Search Console metrics reflect impressions and clicks from Google search engine queries only, completely omitting direct traffic, social media referrals, and non-Google search engines.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.